In [40]:
import pandas as pd
import pickle 
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [41]:
data = pd.read_csv("data.csv")
data = data.sort_values("date").reset_index(drop = True)

In [42]:
data.head()

,date,volume_litres,region,product_type
0,2024-11-04,71.0,West,Small
1,2024-11-05,185.0,West,Medium
2,2024-11-06,117.0,Nort,Small
3,2024-11-07,317.0,Nort,Large
4,2024-11-08,107.0,West,Small


In [43]:
data = pd.DataFrame({"ds":pd.to_datetime(data["date"]), "y":data["volume_litres"]})

In [44]:
split_index = int(len(data) * 0.8)
train_data = data.iloc[:split_index]
print("train:", len(train_data), "test", len(data)- split_index)

train: 584 test 146


In [45]:
final_model = Prophet()
final_model.fit(train_data)

12:23:52 - cmdstanpy - INFO - Chain [1] start processing
12:23:52 - cmdstanpy - INFO - Chain [1] done processing


In [46]:
with open("prophet_model.pkl", "wb") as f:
    pickle.dump(final_model, f)
print("model saved")    

model saved


In [47]:
all_result = []
fold_maes = []
fold_mapes = []

In [48]:
for start in range(split_index , len(data), 7):
    end = min(start + 7 , len(data))
    history = data.iloc[:start]
    test_part = data.iloc[start:end]

    m = Prophet()
    m.fit(history)
    forcast = m.predict(test_part[["ds"]])
    
    actual = test_part["y"].values
    pred = forcast["yhat"].values

    fold_maes.append(mean_absolute_error(actual, pred))
    fold_mapes.append(mean_absolute_percentage_error(actual, pred) * 100)

    result = pd.DataFrame({
        "date" :test_part["ds"].dt.date.values,
        "actual" : actual,
        "prophet_pred" : pred.round(2)
    })
    all_result.append(result)
    
final = pd.concat(all_result)
final.to_csv("backtest_result.csv", index = False)
print("Rows:", len(final), "Folds:", len(fold_maes))   

12:23:55 - cmdstanpy - INFO - Chain [1] start processing
12:23:56 - cmdstanpy - INFO - Chain [1] done processing
12:23:56 - cmdstanpy - INFO - Chain [1] start processing
12:23:57 - cmdstanpy - INFO - Chain [1] done processing
12:23:57 - cmdstanpy - INFO - Chain [1] start processing
12:23:57 - cmdstanpy - INFO - Chain [1] done processing
12:23:58 - cmdstanpy - INFO - Chain [1] start processing
12:23:58 - cmdstanpy - INFO - Chain [1] done processing
12:23:59 - cmdstanpy - INFO - Chain [1] start processing
12:23:59 - cmdstanpy - INFO - Chain [1] done processing
12:24:00 - cmdstanpy - INFO - Chain [1] start processing
12:24:00 - cmdstanpy - INFO - Chain [1] done processing
12:24:00 - cmdstanpy - INFO - Chain [1] start processing
12:24:01 - cmdstanpy - INFO - Chain [1] done processing
12:24:01 - cmdstanpy - INFO - Chain [1] start processing
12:24:01 - cmdstanpy - INFO - Chain [1] done processing
12:24:02 - cmdstanpy - INFO - Chain [1] start processing
12:24:02 - cmdstanpy - INFO - Chain [1]

Rows: 146 Folds: 21


In [50]:
# Average Prophet errors
prophet_mae = sum(fold_maes) / len(fold_maes)
prophet_mape = sum(fold_mapes) / len(fold_mapes)

# Comparison table
comparison = f"""# Baseline vs Prophet

| Model | MAE (liters) | MAPE (%) |
|---|---:|---:|
| Moving Average | 85.19 | 53.55 |
| Prophet | {prophet_mae:.2f} | {prophet_mape:.2f} |
"""

# Save file
with open("comparison.md", "w") as f:
    f.write(comparison)

print("Comparison file created successfully!")


Comparison file created successfully!
